# MSK Annotation Suite — scripts_v5

Refactored from v4. Same lessons, far fewer cells.

**Keep:** ingest diagnostics, sulcus/`true_protocol` handling, ICC, Bland–Altman, pairwise redo, JMP export, planar check.

**Drop:** ML ground-truth exports, consensus/Difficult/Calibration redo, ad-hoc diagnostic one-offs (folded into one coverage diagnostic).

**Safety:** all writes go to `analysis_outputs_v5/`. The live `redo_tracker.xlsx` at the project root is never overwritten.


## 0. Setup

Loads `scripts_v5_lib.py` from this folder (works in Colab after Drive mount, or locally with Drive for Desktop).


In [ ]:
import sys
from pathlib import Path

# Make sure this notebook's folder is on the path (Colab + local)
HERE = Path.cwd()
candidates = [HERE, HERE / "scripts", Path("/content/drive/MyDrive/Current Knee MRI Project Folder/scripts")]
for c in candidates:
    if (c / "scripts_v5_lib.py").exists():
        sys.path.insert(0, str(c))
        print("Using lib from:", c)
        break
else:
    # Fall back: same directory as this notebook when launched from scripts/
    sys.path.insert(0, str(HERE))

import scripts_v5_lib as v5
v5.ensure_packages()
project_path = v5.resolve_project_path()
print("Project:", project_path)
print("v5 outputs:", project_path / v5.OUTPUT_DIR_NAME)
print("Live redo tracker (will NOT be written):", project_path / "redo_tracker.xlsx")
LIVE_MTIME = (project_path / "redo_tracker.xlsx").stat().st_mtime if (project_path / "redo_tracker.xlsx").exists() else None


## 1. Load CSVs + standardize + coverage diagnostic

This is the lesson-learned check: counts per rater × patient × protocol, including separate `sulcus-angle` vs `sulcus-angle-3cm` via `true_protocol` from `groupId` (because `protocolId` is `"sulcus-angle"` for both).


In [ ]:
all_rows, dataframes, rater_keys = v5.load_all_exports(project_path)
dataframes = v5.standardize_all(dataframes, rater_keys)
all_rows = __import__("pandas").concat(dataframes.values(), ignore_index=True)

diag = v5.coverage_diagnostic(all_rows, v5.active_rater_keys(rater_keys))
for name, df in diag.items():
    if hasattr(df, "empty") and not df.empty:
        path = v5.safe_write_path(project_path, "diagnostics", f"{name}.csv")
        df.to_csv(path)
        print("Saved", path)


## 2. Comparison table + ICC / Pearson

**Rater toggle:** edit `ICC_INCLUDE_RATERS` / `ICC_EXCLUDE_RATERS` in the next cell (must be a **list of strings**, not one string).

**Sulcus (v4 parity):** `COLLAPSE_SULCUS_FOR_ICC = True` (default) renames plain `sulcus-angle` → `sulcus-angle-3cm` before ICC, matching v4’s historical ~0.70 sulcus number. Set `False` to keep them separate.


In [ ]:
# ── Choose raters for ICC (edit these) ──────────────────────────────────────
# IMPORTANT: use a list of strings, e.g. ["Danish", "Miyaz"] — not "Danish".
ICC_INCLUDE_RATERS = ["Danish", "Madalyn", "Miyaz", "Parker", "Roy"]  # or None
ICC_EXCLUDE_RATERS = ["Zachary Liu", "Raaga Ramesh", "Justin Lin"]

# Match v4 historical sulcus ICC (~0.70). Set False to analyze protocols separately.
COLLAPSE_SULCUS_FOR_ICC = True

comparison_df, value_cols = v5.build_comparison_df(
    dataframes, rater_keys, collapse_sulcus=COLLAPSE_SULCUS_FOR_ICC
)
icc_summary, pearson_df = v5.compute_icc_and_pearson(
    comparison_df,
    rater_keys,
    include=ICC_INCLUDE_RATERS,
    exclude=ICC_EXCLUDE_RATERS,
)

if len(icc_summary):
    icc_summary.to_csv(v5.safe_write_path(project_path, "icc_summary.csv"), index=False)
if len(pearson_df):
    pearson_df.to_csv(v5.safe_write_path(project_path, "pearson_pairwise.csv"), index=False)


## 3. Bland–Altman + bias summary


In [ ]:
bias_df = v5.bland_altman_and_bias(comparison_df, rater_keys, make_plots=True)
if len(bias_df):
    bias_df.to_csv(v5.safe_write_path(project_path, "bland_altman_bias_summary.csv"), index=False)


## 4. Pairwise redo tracker (DRAFT only)

Writes to `analysis_outputs_v5/redo_tracker/` — **not** the live project-root tracker.


In [ ]:
tracker_df = v5.build_pairwise_redo_tracker(comparison_df, rater_keys, project_path)
v5.assert_live_tracker_untouched(project_path, LIVE_MTIME)


## 5. JMP long export (primary data export)

Refined: separate sulcus protocols, deduped, coverage table printed, saved under `analysis_outputs_v5/` only.


In [ ]:
jmp_long = v5.export_jmp_long(
    comparison_df,
    rater_keys,
    project_path,
    include_single_rater=True,  # full coverage for JMP (not only ≥2-rater matches)
    all_rows=all_rows,
)
v5.assert_live_tracker_untouched(project_path, LIVE_MTIME)


## 6. Planar landmark variability (optional)

In-plane click agreement — useful for diagnosing large landmark outliers. Not an ML export.


In [ ]:
planar_stats = v5.planar_variability_summary(dataframes, rater_keys, project_path)
v5.assert_live_tracker_untouched(project_path, LIVE_MTIME)
print("\nDone. Review outputs in:", project_path / v5.OUTPUT_DIR_NAME)
print("Promote the draft redo tracker only after you approve it.")
